In [ ]:
import pandas as pd
import re

BOOKS_CSV     = 'books.csv'
TAGS_CSV      = 'tags.csv'
BOOK_TAGS_CSV = 'book_tags.csv'


In [ ]:
books     = pd.read_csv(BOOKS_CSV)
tags      = pd.read_csv(TAGS_CSV)
book_tags = pd.read_csv(BOOK_TAGS_CSV)

print('books:', books.shape, '| tags:', tags.shape, '| book_tags:', book_tags.shape)


In [ ]:
book_tags_strong = book_tags[book_tags['count'] >= 7]
print('после count >= 7:', book_tags_strong.shape)


In [ ]:
step1 = book_tags_strong.merge(tags, on='tag_id')
step2 = step1.merge(books, on='goodreads_book_id')

df = step2[['book_id', 'title', 'tag_name', 'count', 'authors']]
print('после merge:', df.shape)
df.head()


In [ ]:
blacklist = [
    'to-read','currently-reading','owned','books-i-own','library','owned-books','to-buy',
    'default','my-books','wish-list','my-library','i-own','own-it','have','borrowed','maybe',
    'abandoned','re-read','did-not-finish','dnf','unfinished','finished','to-re-read',
    'want-to-read','need-to-buy','tbr','must-read','on-hold','own-to-read','unread',
    'gave-up-on','gave-up','want','books-i-have','on-my-shelf','bookshelf','my-bookshelf',
    'personal-library','home-library','mine','couldn-t-finish','didn-t-finish',
    'never-finished','read-more-than-once','listened-to','reread','books-to-buy',
    'on-my-kindle','my-ebooks','on-kindle','to-read-fiction','to-read-non-fiction',
    'to-read-nonfiction','to-read-fantasy','to-read-classics','non-fiction-to-read',
    'classics-to-read',
    'kindle','ebook','audiobook','ebooks','audiobooks','audio','e-book','audible','e-books',
    'audio-books','audio-book','paperback','nook','calibre','kindle-books','hardcover',
    'overdrive','netgalley','signed',
    'read-in-2015','read-in-2014','read-in-2016','read-in-2013','read-in-2012','read-in-2011',
    'read-in-2010','read-in-2009','read-in-2017','read-2015','read-2014','read-2016',
    'read-2013','read-2012','read-2011','read-2010','read-2017','2015-reads','2016-reads',
    '2014-reads','2013-reads','2012-reads','2016-books','2015-books','2014-books','2016-read',
    '2015-read','2014-read','2016-reading-challenge','2015-reading-challenge','2017-reads',
    '5-stars','4-stars','3-stars','5-star','reviewed','arc','first-reads',
    'favorites','favourites','favorite','favorite-books','all-time-favorites',
    'favorite-authors','favorite-series','faves','favs','favourite','my-favorites','loved',
    'meh','recommended','shelfari-favorites','reference','other','general','read-aloud',
    'read-for-school','for-school','school','school-books','college','classroom-library',
    'book-club','bookclub','book-group','book-club-books','book-club-reads','read-in-english',
    'english','american','british','england','usa','uk','europe','america','new-york',
    'translated','british-literature','american-literature','american-lit',
    'fiction','novels','novel','books','series','adult','adult-fiction','general-fiction',
    'literature','literary','literary-fiction','lit','part-of-a-series','stand-alone',
    'standalone','trilogy','finished-series','first-in-series','stories','short-story',
    'short-stories','modern','contemporary','contemporary-fiction','modern-fiction',
    'realistic','realistic-fiction','fiction-historical',
    'young-adult','ya','teen','children','childrens','kids','middle-grade','juvenile',
    'new-adult','youth','high-school','youngadult','ya-fiction','ya-books','ya-lit',
    'ya-fantasy','ya-romance','ya-paranormal','ya-contemporary','young-adult-fiction',
    'young-adult-fantasy','childrens-books','children-s','children-s-books','kids-books',
    'kid-lit','kid-books','childrens-lit','childrens-fiction','children-books',
    'children-s-lit','children-s-literature','children-s-fiction','childrens-literature',
    'childhood','childhood-books','childhood-favorites','childhood-reads','juvenile-fiction',
    'middle-school','picture-books','chapter-books','read-as-a-kid','teen-fiction',
    'female-author','female-authors','women','women-s-fiction',
    'library-books','library-book',
]

df_filter = df[~df['tag_name'].isin(blacklist)]
print('после чёрного списка:', df_filter.shape)
print('осталось тегов:', df_filter['tag_name'].nunique())


In [ ]:
df_top = (df_filter
          .sort_values('count', ascending=False)
          .groupby('book_id')
          .head(7))

print('после top-7:', df_top.shape)
print('уникальных книг:', df_top['book_id'].nunique(),
      '| уникальных тегов:', df_top['tag_name'].nunique())


In [ ]:
author_tokens = set()
for a in df_top['authors'].dropna().unique():
    for name in str(a).split(','):
        parts = [p for p in re.split(r'[^a-zA-Zа-яА-Я]+', name.strip().lower()) if len(p) > 2]
        if parts:
            author_tokens.add('-'.join(parts))
            author_tokens.add('-'.join(reversed(parts)))
            if len(parts) >= 2:
                author_tokens.add(parts[-1])
                author_tokens.add(parts[0] + '-' + parts[-1])
                author_tokens.add(parts[-1] + '-' + parts[0])

EXPLICIT_JUNK = {
    '1001','1001-books','1001-books-to-read-before-you-die','1001-books-to-read',
    '1001-to-read','1001-import','1001-books-you-must-read-before-you',
    'rory-gilmore-reading-challenge','newbery','newbery-medal','newbery-honor',
    'summer-reading','reading-challenge','banned-books','pulitzer','man-booker',
    'booker-prize','nobel','oprah-s-book-club','oprah','bbc-big-read',
    'kindle-unlimited','audible-uk','scribd','goodreads','goodreads-giveaway',
    'first-reads-giveaway','giveaway','freebie','free','free-on-kindle','amazon',
    'novella','novellas','play','plays','poem','poems','anthology','anthologies',
    'omnibus','boxed-set','collection','collections','picture-book',
    'children-s-picture-books','textbook','textbooks','manual','guide',
    'academic','required-reading','summer','fall','winter','spring',
    'middle-earth','midkemia','mistborn','oz','dexter','delirium','dee','abarat',
    'narnia','discworld','westeros','hogwarts','panem','pern','shannara','dune',
    'wheel-of-time','malazan','cosmere','riyria','carpathians','caster-chronicles',
    'hush-hush','heroes-of-olympus','clifton-chronicles','starcrossed',
    'harry-potter','twilight','hunger-games','percy-jackson','divergent',
    'lord-of-the-rings','game-of-thrones','outlander','sookie-stackhouse',
    'vampire-academy','shadowhunters','mortal-instruments','maze-runner',
    'on-writing','debut','nostalgia','saw-the-movie','movie','movies','tv',
    'mhc','richard-castle','girl-online','firelight',
}
SURNAME_HUBS = {
    'king','patterson','koontz','grisham','roberts','christie','austen','tolkien',
    'rowling','sparks','steel','cornwell','evanovich','child','gaiman','pratchett',
    'sanderson','martin','riordan','meyer','picoult',
}

def is_junk(tag):
    t = str(tag).strip().lower()
    if not t or t in EXPLICIT_JUNK or t in SURNAME_HUBS or t in author_tokens:
        return True
    if not re.fullmatch(r"[a-z0-9\-'&\. ]+", t):
        return True
    if re.search(r'(19|20)\d{2}', t):
        return True
    if re.match(r'^\d+[\-_]', t) or re.search(r'[\-_]\d+$', t):
        return True
    return False

junk = {tg for tg in df_top['tag_name'].unique() if is_junk(tg)}
clean = df_top[~df_top['tag_name'].isin(junk)]

freq = clean.groupby('tag_name')['book_id'].nunique()
clean = clean[clean['tag_name'].isin(set(freq[freq >= 5].index))]

print('итог связей:', len(clean),
      '| тегов:', clean['tag_name'].nunique(),
      '| книг:', clean['book_id'].nunique())


In [ ]:
edges = pd.DataFrame({
    'source': 'b' + clean['book_id'].astype(str),
    'target': 't_' + clean['tag_name'].astype(str),
    'weight': clean['count'],
    'tag': clean['tag_name'],
})

book_nodes = clean[['book_id', 'title', 'authors']].drop_duplicates(subset='book_id').copy()
book_nodes['id']      = 'b' + book_nodes['book_id'].astype(str)
book_nodes['label']   = book_nodes['title']
book_nodes['type']    = 'book'
book_nodes['author']  = book_nodes['authors']
book_nodes['n_books'] = 1
book_nodes = book_nodes[['id', 'label', 'type', 'author', 'n_books']]

tag_freq = clean.groupby('tag_name')['book_id'].nunique().reset_index(name='n_books')
tag_nodes = pd.DataFrame({
    'id':      't_' + tag_freq['tag_name'],
    'label':   tag_freq['tag_name'],
    'type':    'tag',
    'author':  '',
    'n_books': tag_freq['n_books'],
})

nodes = pd.concat([book_nodes, tag_nodes], ignore_index=True)

print('РЁБРА:', edges.shape, '| УЗЛЫ:', nodes.shape,
      '(книг', (nodes.type == 'book').sum(), ', тегов', (nodes.type == 'tag').sum(), ')')

missing = (set(edges['source']) | set(edges['target'])) - set(nodes['id'])
print('потерянных узлов:', len(missing))


In [ ]:
edges.to_csv('edges_bipartite.csv', index=False)
nodes.to_csv('nodes_bipartite.csv', index=False)
print('Сохранено: edges_bipartite.csv, nodes_bipartite.csv')

tag_freq.sort_values('n_books', ascending=False).head(15)
